# 1. Загрузка и очистка данных

Задача проекта: по данным одной госпитализации предсказать, вернётся ли пациент с диабетом в больницу в течение 30 дней.

Функции лежат в `src/data.py` и `src/features.py`, здесь мы их вызываем.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # чтобы работал импорт из папки src

import pandas as pd

from src import config, data, features

## Загрузка

Пропуски в этом файле записаны знаком `?`. `load_raw` читает его как пропуск.

In [2]:
raw = data.load_raw()
raw.shape

(101766, 50)

In [3]:
raw.head(3)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


## Память

`optimize_dtypes` хранит текст как `category`, а числа — в самом маленьком подходящем типе.

In [4]:
optimized = data.optimize_dtypes(raw)
print("До:   ", data.memory_mb(raw), "МБ")
print("После:", data.memory_mb(optimized), "МБ")

До:    48.51 МБ
После: 5.85 МБ


`load_in_chunks` читает файл кусками по 20 000 строк. Так можно читать файлы, которые не помещаются в память целиком.

In [5]:
chunked = data.load_in_chunks(chunksize=20000)
print(len(chunked), "строк,", data.memory_mb(chunked), "МБ")

101766 строк, 5.85 МБ


## Аудит качества

`audit` считает факты о данных до очистки и ничего не меняет.

In [6]:
audit = data.audit(raw)
pd.Series({key: value for key, value in audit.items() if not isinstance(value, dict)})

rows                             101766
columns                              50
unique_patients                   71518
patients_with_repeat_visits       16773
rows_from_repeat_patients         47021
duplicate_encounter_ids               0
gender_unknown_rows                   3
expired_or_hospice_rows            2423
expired_but_marked_readmitted        86
dtype: int64

Пропуски по столбцам, %:

In [7]:
pd.Series(audit["missing_percent"])

weight               96.86
max_glu_serum        94.75
A1Cresult            83.28
medical_specialty    49.08
payer_code           39.56
race                  2.23
diag_3                1.40
diag_2                0.35
diag_1                0.02
dtype: float64

Ответ `readmitted`: `NO` — не вернулся, `>30` — вернулся позже 30 дней, `<30` — раньше 30 дней.

In [8]:
pd.Series(audit["target_distribution"])

NO     54864
>30    35545
<30    11357
dtype: int64

Коды выписки «умер» и «хоспис» берём из справочника. Такие пациенты вернуться не могут.

In [9]:
discharge = data.load_mapping()["discharge_disposition_id"]
discharge[discharge["id"].isin(config.EXPIRED_OR_HOSPICE)]

,id,description
10,11,Expired
12,13,Hospice / home
13,14,Hospice / medical facility
18,19,"Expired at home. Medicaid only, hospice."
19,20,"Expired in a medical facility. Medicaid only, ..."
20,21,"Expired, place unknown. Medicaid only, hospice."


## Очистка

| Правило | Причина |
|---|---|
| удалить визиты «умер / хоспис» | вернуться не могут |
| удалить строки без пола | их всего 3 |
| удалить `weight` и почти постоянные столбцы | 97% пропусков или одно значение |
| `A1Cresult`, `max_glu_serum`: пропуск → `NotMeasured` | анализ не делали — это информация |
| `medical_specialty`, `payer_code`, `race`: пропуск → `Unknown` | пропусков слишком много |

In [10]:
clean = data.clean(raw)
print("Строк:   ", len(raw), "->", len(clean))
print("Столбцов:", raw.shape[1], "->", clean.shape[1])

Строк:    101766 -> 99340
Столбцов: 50 -> 44


In [11]:
print("Удалены:", [col for col in raw.columns if col not in clean.columns])
print("Добавлены:", [col for col in clean.columns if col not in raw.columns])

Удалены: ['weight', 'acetohexamide', 'tolbutamide', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
Добавлены: ['diag_1_group', 'diag_2_group', 'diag_3_group', 'readmit_30', 'total_prior_visits', 'age_num']


Пример почти постоянного столбца: лекарство `examide` не назначали никому.

In [12]:
raw["examide"].value_counts()

examide
No    101766
Name: count, dtype: int64

## Новые признаки

**Группы диагнозов.** ICD-9 — международные коды болезней. Разных кодов сотни, поэтому `map_icd9` сводит их к 9 группам.

In [13]:
for code in ["428", "250.83", "486", "V57", "820", None]:
    print(code, "->", features.map_icd9(code))

428 -> Circulatory
250.83 -> Diabetes
486 -> Respiratory
V57 -> Other
820 -> Injury
None -> Missing


In [14]:
print("Кодов было:", raw["diag_1"].nunique())
clean["diag_1_group"].value_counts()

Кодов было: 716


diag_1_group
Circulatory        29680
Other              17793
Respiratory        13934
Digestive           9333
Diabetes            8661
Injury              6851
Genitourinary       5002
Musculoskeletal     4935
Neoplasms           3131
Missing               20
Name: count, dtype: int64

**Возраст числом** — середина интервала. **Все прошлые обращения** — сумма трёх счётчиков.

In [15]:
clean[["age", "age_num", "number_outpatient", "number_emergency", "number_inpatient", "total_prior_visits"]].head()

,age,age_num,number_outpatient,number_emergency,number_inpatient,total_prior_visits
0,[0-10),5,0,0,0,0
1,[10-20),15,0,0,0,0
2,[20-30),25,2,0,1,3
3,[30-40),35,0,0,0,0
4,[40-50),45,0,0,0,0


**Ответ `readmit_30`:** 1 — вернулся раньше 30 дней, 0 — всё остальное.

In [16]:
pd.crosstab(clean["readmitted"], clean["readmit_30"])

readmit_30,0,1
readmitted,,
<30,0,11314
>30,35502,0
NO,52524,0


In [17]:
print("Доля вернувшихся раньше 30 дней:", round(clean["readmit_30"].mean() * 100, 1), "%")
print("Признаков в модели:", len(features.FEATURES))

Доля вернувшихся раньше 30 дней: 11.4 %
Признаков в модели: 36


## Сохранение

Parquet — сжатый формат таблиц, который хранит типы столбцов.

In [18]:
data.optimize_dtypes(clean).to_parquet(config.CLEAN_PARQUET, index=False)
print("CSV:    ", round(config.RAW_CSV.stat().st_size / 1024**2, 1), "МБ")
print("Parquet:", round(config.CLEAN_PARQUET.stat().st_size / 1024**2, 1), "МБ")

CSV:     18.3 МБ
Parquet: 2.6 МБ


## Итог

- 101 766 визитов, 71 518 пациентов; у 16 773 пациентов больше одного визита.
- Удалено 2 426 строк и 12 столбцов, добавлено 6 столбцов. Осталось 99 340 строк.
- Раньше 30 дней вернулись 11,4% — классы несбалансированы.
- Память: 48,5 МБ → 5,9 МБ после сжатия типов.